# AI Simplified Lab — Remotion render worker (Colab)

Executes one `CompositionJob` bundle with the locked Remotion runtime and
uploads the MP4 plus reports. This notebook invents nothing: every timing,
asset, and transition comes from the job bundle. Set `JOB_ZIP` below,
then Runtime → Run all.

In [ ]:
JOB_ZIP = "/content/drive/MyDrive/AI-Simplified-Video-Factory/jobs/proj_3e27bd7a_remotion-bundle.zip"  # <-- set this
RESULTS_DIR = "/content/drive/MyDrive/AI-Simplified-Video-Factory/results"
WORKDIR = "/content/remotion_work"
REPO = "/content/factory"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil, subprocess

def sh(argv):
    print('$', ' '.join(argv), flush=True)
    proc = subprocess.run(argv, capture_output=True, text=True)
    print((proc.stdout or '')[-2000:])
    print((proc.stderr or '')[-2000:])
    assert proc.returncode == 0, f"command failed: {argv[0]}"
    return proc

if shutil.which('node') is None:
    sh(['bash', '-lc', 'curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - && sudo apt-get install -y nodejs'])
sh(['node', '--version'])
sh(['npm', '--version'])
sh(['ffmpeg', '-version'])

In [ ]:
import shutil, pathlib

assert pathlib.Path(JOB_ZIP).is_file(), f'job bundle not found: {JOB_ZIP}'
shutil.rmtree(WORKDIR, ignore_errors=True)
pathlib.Path(WORKDIR).mkdir(parents=True)
# The factory repo (composition/remotion adapter + worker script) must be
# available: clone it here, or upload scripts/remotion_worker.py alongside.
# Example: !git clone <factory-repo-url> /content/factory
print('workdir ready:', WORKDIR)

In [ ]:
import datetime, json, pathlib

started = datetime.datetime.now(datetime.timezone.utc).isoformat()
proc = subprocess.run(
    ['python3', f'{REPO}/scripts/remotion_worker.py', '--bundle', JOB_ZIP,
     '--workdir', WORKDIR, '--sync-dir', f'{RESULTS_DIR}/' + pathlib.Path(JOB_ZIP).stem.replace('_remotion-bundle', ''),
     '--install-node'],
    capture_output=True, text=True,
)
print(proc.stdout[-4000:])
print(proc.stderr[-4000:])
print('worker exit code:', proc.returncode)
assert proc.returncode == 0, 'worker failed: see worker_result.json in the results dir'

In [ ]:
import json, glob

results = sorted(glob.glob(f'{RESULTS_DIR}/*/worker_result.json'))
assert results, 'no worker results synced'
latest = max(results, key=lambda p: open(p).read().find('completed'))
print(json.dumps(json.load(open(latest)), indent=2))